# 01 — Exploratory Data Analysis

**Goal of this notebook:** understand the data well enough to make *defensible* cleaning and modelling decisions.
We won't train any models here. Every decision we make later (what to drop, what to rename, how to split) should trace back to evidence in this notebook.

**Sections**
1. Setup & load
2. Structure: dtypes, missing values, panel shape
3. The target: `has_claim`
4. Checking column meanings (does the data match the dictionary?)
5. Leakage audit
6. Exposure
7. Claim rate by feature
8. Summary of decisions

> 💡 **How to use this notebook:** read each explanation cell *before* running the code under it, and write a one-line note in the **📝 Your notes** cells. Those notes become your README and your answers when someone asks "why did you do that?"

## 1. Setup & load

**Where's the data?** The CSV (95 MB) isn't in GitHub; it's too big and it isn't ours to redistribute. Upload `motor_portfolio.csv` to your Google Drive once (e.g. `My Drive/motor-insurance/`), and this cell mounts Drive so Colab can read it.

The cell also works when run locally (it falls back to `data/raw/`), so the notebook isn't tied to Colab.

In [ ]:
import os

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA_PATH = "/content/drive/MyDrive/motor-insurance/motor_portfolio.csv"  # 👈 change if you saved it elsewhere
except ImportError:
    DATA_PATH = "../data/raw/motor_portfolio.csv"  # running locally

print("Reading from:", DATA_PATH, "| exists:", os.path.exists(DATA_PATH))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 60)

# The file is semicolon-separated (common in European exports, where the comma is the decimal mark)
df = pd.read_csv(DATA_PATH, sep=";")
print(df.shape)
df.head()

## 2. Structure: dtypes, missing values, panel shape

**Why check dtypes first?** A numeric column read as text (or a category stored as a number) silently breaks everything downstream. For example, the model would treat `seats = 5` as "more" than `seats = 4`, which is fine, but it would do the same with an ID column, which is not.

In [ ]:
df.dtypes

**Missing values.** For each column we need a decision: drop the rows, fill them in (impute), or keep "missing" as its own category.
The right answer depends on *how many* values are missing and *whether missingness itself carries information*.

In [ ]:
missing = df.isna().sum()
missing = missing[missing > 0].to_frame("n_missing")
missing["pct"] = (missing["n_missing"] / len(df) * 100).round(3)
missing

**Panel shape.** This is *panel data*: the same insured appears in several years. That matters for how we split train and test (Section 8 / the project plan).

In [ ]:
print("Rows:", len(df))
print("Unique insureds:", df["insured_id"].nunique())
print("\nRows per insured:")
print(df.groupby("insured_id").size().value_counts().sort_index())
print("\nRows per year:")
print(df["year"].value_counts().sort_index())
print("\nDuplicate (insured_id, year) pairs:", df.duplicated(["insured_id", "year"]).sum())

> 📝 **Your notes:** What surprised you? Which missing-value columns look worth investigating?

## 3. The target: `has_claim`

We predict **whether a policy has at least one claim** in the period, not how many claims it has.
*Why binary?* It's simpler to model, to evaluate, and to explain in a demo, and it's still a real underwriting question. Modelling claim *counts* (a Poisson model) is listed as future work.

**Why look at the base rate?** It tells us how imbalanced the problem is, and it's our dumbest baseline: a model that just predicts "14% for everyone" is the bar we need to clear.

In [ ]:
df["has_claim"] = (df["total_claims"] > 0).astype(int)

print("Overall claim rate:", round(df["has_claim"].mean(), 4))
print("\nClaim rate by year:")
print(df.groupby("year")["has_claim"].mean().round(4))

**What to look for:** Is the rate stable across years? If it drifts, a model trained on older years will be slightly miscalibrated on newer ones, and that's a reason to test on the *latest* year, as a real insurer would.

> 📝 **Your notes:**

## 4. Checking column meanings

**Why?** Data dictionaries are written by people and are often wrong or out of date. If we trust a mislabeled column, the model can still work, but *our explanation of it* will be wrong, and in insurance, explanations matter (regulators, underwriters, and customers all ask "why this price?").

The dictionary says:
- `vehicle_age`: "Age of the insured vehicle, measured in years."
- `age_driving_licence`: "Year in which the insured obtained the driving licence."

Let's test both claims against the data.

### 4a. `age_driving_licence`: is it a year?
If it's a calendar year, we'd expect values like 1985 or 2010.

In [ ]:
df["age_driving_licence"].describe()

### 4b. `vehicle_age`: two competing hypotheses

| Hypothesis | Prediction we can test |
|---|---|
| **H1: it really is vehicle age** (older values might be collectible or imported cars) | Weak link to the driver's age; very old cars are rare; old cars are worth more *or* less than newer ones; collectible brands dominate the oldest group |
| **H2: it is actually years licensed** (`driver_age − age_driving_licence`) | The column equals that subtraction almost everywhere |

A good analyst writes down what each hypothesis predicts **before** looking, so the evidence decides rather than our expectations.

In [ ]:
d = df.dropna(subset=["vehicle_age", "age_driving_licence", "vehicle_value"]).copy()
d["years_licensed_calc"] = d["driver_age"] - d["age_driving_licence"]
gap = d["vehicle_age"] - d["years_licensed_calc"]

print("vehicle_age minus (driver_age - age_driving_licence):")
print(gap.value_counts().head(6))
print("\nShare within ±1 year:", round((gap.abs() <= 1).mean(), 4))

In [ ]:
print("Correlation with driver_age:   ", round(d["vehicle_age"].corr(d["driver_age"]), 3))
print("Correlation with vehicle_value:", round(d["vehicle_age"].corr(d["vehicle_value"]), 3))
print("Share of 'vehicles' aged 40+:  ", round((d["vehicle_age"] >= 40).mean(), 3))

In [ ]:
# H1 test: are the oldest 'vehicles' worth more (collectibles) or a different brand mix?
age_bins = pd.cut(d["vehicle_age"], [-1, 5, 10, 20, 30, 40, 70])
print("Median vehicle_value by vehicle_age bucket:")
print(d.groupby(age_bins, observed=True)["vehicle_value"].median().round(0))

old = d[d["vehicle_age"] >= 40]
compare = pd.DataFrame({
    "age 40+": old["vehicle_brand"].value_counts(normalize=True).head(8),
    "all":     d["vehicle_brand"].value_counts(normalize=True).head(8),
}).round(3)
compare

In [ ]:
# The sharpest test: drivers licensed THIS year (years licensed = 0).
# Under H1 their cars could be any age. Under H2 vehicle_age must be 0.
new_drivers = d[d["years_licensed_calc"] == 0]
print("Drivers licensed this year:", len(new_drivers))
print(new_drivers["vehicle_age"].value_counts().head())

In [ ]:
# Visual check: if H2 is true, the points sit on the diagonal
sample = d.sample(5000, random_state=42)
plt.figure(figsize=(5, 5))
plt.scatter(sample["years_licensed_calc"], sample["vehicle_age"], s=3, alpha=0.3)
plt.plot([0, 70], [0, 70], "r--", label="y = x")
plt.xlabel("driver_age − age_driving_licence")
plt.ylabel("vehicle_age (as labelled)")
plt.legend()
plt.title("Is 'vehicle_age' really years licensed?")
plt.show()

> 📝 **Your notes / conclusion:** Which hypothesis does the evidence support, and which single test convinced you most?
> Whatever you conclude, we **don't edit the raw file**. A rename (if any) happens in the cleaning code, so it's documented and reversible.

## 5. Leakage audit

**Target leakage** means a feature contains information that wouldn't be available *at the moment we make the prediction*. A leaky model scores brilliantly in testing and fails in production.

**The question for every column:** *"Would an underwriter know this on the day the policy starts?"*

In [ ]:
claims_cols   = [c for c in df.columns if c.endswith("_claims")]
incurred_cols = [c for c in df.columns if c.endswith("_incurred")]
premium_cols  = [c for c in df.columns if c.endswith("_premium")]

print(f"{len(claims_cols)} claim-count columns:", claims_cols)
print(f"\n{len(incurred_cols)} incurred-cost columns:", incurred_cols)
print(f"\n{len(premium_cols)} premium columns:", premium_cols)

- **Claims and incurred columns**: these *are* the outcome. Obvious leakage, so they're dropped (except `total_claims`, which builds the target).
- **`policy_status`**: the dictionary says it's "active or cancelled *at the time of analysis*", i.e. after the fact. Let's see whether it relates to the outcome:

In [ ]:
print(df.groupby("policy_status")[["has_claim", "total_exposure"]].mean().round(3))

Cancelled policies also have much lower exposure, because they ended early. Cancellation happens *during or after* the period, so we can't know it on day one. **→ Drop.**

**Premiums** are trickier. They're set before the policy starts, so they aren't strictly leakage, but let's check whether they're the *annual* price or the *earned* (pro-rated) amount:

In [ ]:
print("Correlation total_premium vs total_exposure:", round(df["total_premium"].corr(df["total_exposure"]), 3))

A strong correlation with exposure suggests the premiums are **earned**, i.e. scaled by time on risk. We'll keep premiums **out of the model** and use them later as a benchmark: *can our model rank risk better than the insurer's own price?*

> 📝 **Your notes:**

## 6. Exposure

**Exposure** = the fraction of a year the policy was active. A policy active for 3 months has had less *time* to produce a claim, so its claim rate should be lower even if the driver is identical.

**Why this matters:** if we ignore exposure, the model will "learn" that whatever correlates with short policies (e.g. new business) is low-risk, when those policies simply had less time to claim.

In [ ]:
print(df["total_exposure"].describe())
print("\nZero-exposure rows:", (df["total_exposure"] == 0).sum())
print("Exposure > 1:", (df["total_exposure"] > 1).sum())
print("Identical to liability_exposure:", (df["total_exposure"] == df["liability_exposure"]).all())

In [ ]:
exp_bins = pd.cut(df["total_exposure"], [0, 0.25, 0.5, 0.75, 0.99, 1.0])
by_exp = df.groupby(exp_bins, observed=True)["has_claim"].agg(["size", "mean"]).round(3)
print(by_exp)

by_exp["mean"].plot(kind="bar", title="Claim rate by exposure", ylabel="claim rate")
plt.show()

**Decision options:**
1. *Filter out* short policies: simple, but it throws away data and changes the population.
2. *Use exposure as a feature*: the model learns the time effect, and in the API we set exposure = 1.0 to get "probability of a claim over a full year". ✅ This is our choice.
3. *Poisson model with an exposure offset*: the actuarial gold standard. Future work.

> 📝 **Your notes:**

## 7. Claim rate by feature

**Why?** Before any modelling, we want intuition: which features seem to move risk, and in which direction? Later, when SHAP tells us what the model learned, we'll compare it against this. If they disagree, something's wrong.

**Caution:** these are *univariate* views (one feature at a time). A feature can look important here only because it's correlated with another one (e.g. exposure). The model sorts that out; this is just a first look.

In [ ]:
overall = df["has_claim"].mean()
cat_cols = ["policy_type", "business_type", "payment_frequency", "bonus_score",
            "fuel_type", "municipality_type", "circulation_area", "seats"]

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
for ax, col in zip(axes.flat, cat_cols):
    rates = df.groupby(df[col].fillna("MISSING"))["has_claim"].mean().sort_values()
    rates.plot(kind="barh", ax=ax, title=col)
    ax.axvline(overall, color="red", linestyle="--", linewidth=1)
plt.suptitle("Claim rate by category (red line = overall rate)")
plt.tight_layout()
plt.show()

In [ ]:
num_cols = ["driver_age", "vehicle_age", "age_driving_licence", "vehicle_value", "power_to_weight_ratio"]

fig, axes = plt.subplots(1, 5, figsize=(22, 4))
for ax, col in zip(axes, num_cols):
    bins = pd.qcut(df[col], 10, duplicates="drop")  # deciles: equal-sized groups
    df.groupby(bins, observed=True)["has_claim"].mean().plot(ax=ax, marker="o", title=col)
    ax.axhline(overall, color="red", linestyle="--", linewidth=1)
    ax.tick_params(axis="x", rotation=90)
plt.tight_layout()
plt.show()

**Vehicle brand** has many categories. Rare categories give noisy estimates, so we'll likely group small brands into "OTHER".

In [ ]:
brand = df.groupby("vehicle_brand")["has_claim"].agg(["size", "mean"]).sort_values("size", ascending=False)
print("Number of brands:", len(brand))
print("Brands with < 1,000 rows:", (brand["size"] < 1000).sum())
brand.head(20).round(3)

**Prior-year claims.** Past claims are one of the strongest predictors in real insurance pricing. We can build this feature because the data is a panel, but it must use **only the previous year**, never the current one.

In [ ]:
df = df.sort_values(["insured_id", "year"])
df["prev_year_claim"] = df.groupby("insured_id")["has_claim"].shift(1)
print(df.groupby(df["prev_year_claim"].map({0: "no", 1: "yes"}).fillna("no_history"))["has_claim"]
        .agg(["size", "mean"]).round(3))

> 📝 **Your notes:** Which 3 features look most predictive? Do their directions make sense from your adjuster experience?

## 8. Summary of decisions

Fill this in as you go. It becomes the "Data & cleaning" section of the README.

| Topic | Decision | Evidence (section) |
|---|---|---|
| Target | `has_claim = total_claims > 0` | §3 |
| `vehicle_age` meaning | | §4b |
| `age_driving_licence` meaning | | §4a |
| Claims / incurred columns | Drop (they are the outcome) | §5 |
| `policy_status` | | §5 |
| Premiums | | §5 |
| Exposure | | §6 |
| Missing values | | §2 |
| Brand grouping | | §7 |
| Prior-year claim feature | | §7 |
| Train/test split | Train 2022–23, test 2024 (see plan) | §3 drift |